## Text input

https://platform.openai.com/docs/models

In [31]:
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [29]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The capital of The Moon is Selene Prime.

A quick snapshot:
- Location: In the Arcadia Basin near the equator, straddling a line between mare plains and the highlands, at the twilight edge where sunrise drapes the city in pale gold.
- Government: The Lunar Concord seats its administration here, with the Lunar Parliament convening in the Crescent Forum and the executive council housed in the Lumen Spire.
- Architecture: A blend of lunar basalt and self-healing transparent polymers. Domed districts float above glassy plains, connected by gravity-lift pylons and a radiant energy grid fed by the nearby solar ring.
- Economy: Helium-3 mining, water-ice refinement, high-density solar energy, and a booming research-and-development sector that exports lunar science and tech.
- Landmarks:
  - Lumen Spire: A 3-kilometer-tall solar tower and city hub.
  - Crescent Gate: The monumental ceremonial entrance to the capital district.
  - Sea of Serenity Observatory: A vast dome offering Earth-like ski

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x10abd5540>, 'last_modified': datetime.datetime(2026, 4, 8, 10, 50, 4, 433000, tzinfo=datetime.timezone.utc)},)


In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [7]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Nice image to anchor a city in. Here’s a full portrait of the capital that fits what you’re seeing.

Name and place
- Horizon Spire, the capital of the Caelian Confederacy (aka the Caelis Republic) on the rocky world Caelis. It sits in the Crestline Basin, a high desert amphitheater ringed by jagged basalt and glacial ridges. A blue gas giant looms on the horizon, with moons skimming the sky, giving the city a constant sense of being watched from above.

Government and society
- Horizon Spire is both political capital and cultural heart. The government is a rotating council known as the Beacon, composed of senior architects, engineers, and guild-masters who design the city’s long-term plans. A separate judiciary and a public council of guilds ensure representation for miners, farmers, scholars, and artificers.
- The people value foresight, sustainability, and elegance in function. Virtuosi of memory, who preserve histories in crystalline data-lattices, are as celebrated as engineers wh

## Audio input

In [23]:
import sounddevice as sd
print(sd.query_devices())

  0 DELL S2722QC, Core Audio (0 in, 2 out)
  1 iPhone XR do Henrique Microphone, Core Audio (1 in, 0 out)
> 2 CORSAIR VIRTUOSO USB Gaming Headset, Core Audio (1 in, 0 out)
< 3 CORSAIR VIRTUOSO USB Gaming Headset, Core Audio (0 in, 2 out)
  4 USB PnP Audio Device, Core Audio (0 in, 2 out)
  5 USB PnP Audio Device, Core Audio (1 in, 0 out)
  6 MacBook Pro Microphone, Core Audio (1 in, 0 out)
  7 MacBook Pro Speakers, Core Audio (0 in, 2 out)


In [22]:
sd.default.device = (2, 3)

In [24]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 10  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 100/100 [00:10<00:00,  9.51it/s]


Done.


In [25]:
from IPython.display import Audio, display
display(Audio(data=base64.b64decode(aud_b64), autoplay=True))

In [33]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Cats are small, carnivorous mammals that are domesticated and kept as pets by millions of people around the world. Known scientifically as Felis catus, they have been valued by humans for their companionship, as well as their ability to hunt vermin.

Cats are highly agile and have strong, flexible bodies. Their retractable claws, sharp teeth, and acute senses of sight and hearing make them excellent hunters. They are also known for their grooming habits and tend to keep themselves very clean.

Cats communicate through a variety of vocalizations, body language, and purring, which is often seen as a sign of contentment. They are generally independent animals, although their personalities can vary widely depending on breed and individual traits.

As for breeds, cats come in a wide range—from the short-haired domestic shorthair to the long-haired Persian, and more exotic breeds like the Sphynx or Bengal.

Cats have also been featured in mythology, folklore, and popular culture, often being